In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#   "microcal[ndv-jup] @ git+https://github.com/fdrgsp/microcal",
# ]
# ///

In [12]:
import ndv

from microcal import ChromaticShiftCorrector, generate_beads_image

In [13]:
# generate a 2-channel synthetic beads image
beads_img, _ = generate_beads_image(
    n_channels=2,
    shape=(512, 512),
    n_beads=50,
    bead_sigma=2,
    bead_intensity=60.0,
    bit_depth=16,
    offset=100,
    shifts=[(0, 0), (1.5, -2.5)],
    rotations=[0, 5],
    scales=[(1, 1), (1.05, 0.95)],
    snr=8,
    seed=42,
)

# visualize the synthetic beads image with ndv
ndv.imshow(
    beads_img,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
# measure the chromatic shift — all detection parameters go here
csc = ChromaticShiftCorrector()
results = csc.measure(
    beads_img,
    reference_channel=0,
    smooth_sigma=3,
    min_distance=2,
    threshold_rel=0.5,
    match_max_distance=50,
    min_pairs=2,
    subpixel_refine=True,
    refine_radius=2,
    verbose=True,
)

microcal._chromatic_shift_corrector | INFO | ch0 (reference): 50 beads detected
microcal._chromatic_shift_corrector | INFO | ch1: 48 beads detected | coarse shift (row, col) = [10.6   8.97]
microcal._chromatic_shift_corrector | INFO | ch1: 41 bead pairs matched (max_distance=50.0px)
microcal._chromatic_shift_corrector | INFO | ch1: fit RMS = 0.322px  transform =
[[  1.04896762   0.08309219 -31.29314132]
 [ -0.09190656   0.94869429  34.92256455]
 [  0.           0.           1.        ]]


In [17]:
results.detection_image.shape

(4, 512, 512)

In [ ]:
# visualize the detected beads in the first (reference) channel (beabs + masks)
ch1_det = results.detection_image[:2, :, :]
# in this image, 0 is the reference channel, and 1 is the beads mask
ndv.imshow(
    ch1_det,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "gray"}},
)

RFBOutputContext()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# visualize the detected beads in the second channel
ch2_det = results.detection_image[2:4, :, :]
# in this image, 2 is the second channel, and 3 is the beads mask
ndv.imshow(
    ch2_det,
    channel_mode="composite",
    luts={0: {"cmap": "magenta"}, 1: {"cmap": "gray"}},
)

In [ ]:
# visualize the matched bead pairs between the two channels
ndv.imshow(results.pairs_image.astype("uint16"), default_lut={"cmap": "glasbey"})

In [ ]:
# validate: re-detects beads on the corrected bead image and reports residuals
val = csc.validate()

In [ ]:
# apply the correction to a sample image
# (here we reuse the bead image for demonstration)
image_corr = csc.apply(image_or_stack=beads_img, crop=True)
ndv.imshow(
    image_corr,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)

In [ ]:
# you can also save the calibration parameters and transform to a JSON file that can be
# loaded for later use without needing to re-run the measurement step
csc.save("calibration.json")

In [ ]:
# apply-only workflow: load a saved calibration without re-running measure()
csc2 = ChromaticShiftCorrector.from_json("calibration.json")
image_corr2 = csc2.apply(image_or_stack=beads_img, crop=True)
ndv.imshow(
    image_corr2,
    channel_mode="composite",
    luts={0: {"cmap": "green"}, 1: {"cmap": "magenta"}},
)